# Deteksi Nama File Identik

Notebook ini memindai seluruh file pada drive yang dipilih, termasuk semua subfolder dan format. Hasil nama duplikat menampilkan nama file, path lengkap, dan ukuran file, diurutkan dari ukuran terbesar. Hasil dapat disimpan sebagai CSV.

Nama dibandingkan beserta ekstensinya. Secara default, perbedaan huruf besar/kecil dianggap sama (misalnya `Laporan.PDF` dan `laporan.pdf`).

## 1. Konfigurasi drive

Ubah `TARGET_DIR` ke root drive yang ingin diperiksa, misalnya `Path("D:/")` atau `Path("E:/")`. Notebook memeriksa semua format file secara rekursif.

In [ ]:
from pathlib import Path

# Ganti D:/ dengan huruf drive HDD yang ingin dipindai, misalnya E:/
TARGET_DIR = Path("D:/")

# True berarti nama dengan kapitalisasi berbeda juga dianggap identik.
CASE_INSENSITIVE = True

# True untuk menyimpan tabel hasil ke file CSV.
EXPORT_CSV = False
OUTPUT_CSV = Path.cwd() / "hasil_deteksi_nama_file_identik.csv"

## 2. Pindai folder dan kelompokkan nama yang sama

In [ ]:
import os
import pandas as pd

if not TARGET_DIR.exists() or not TARGET_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan atau bukan folder: {TARGET_DIR}")

files = []
scan_errors = []

def record_walk_error(error):
    scan_errors.append({"path": getattr(error, "filename", ""), "error": str(error)})

for current_dir, _, filenames in os.walk(TARGET_DIR, onerror=record_walk_error):
    for filename in filenames:
        path = Path(current_dir) / filename
        try:
            size_bytes = path.stat().st_size
        except OSError as error:
            scan_errors.append({"path": str(path), "error": str(error)})
            continue
        files.append({"path": path, "size_bytes": size_bytes})

def comparison_key(filename):
    return filename.casefold() if CASE_INSENSITIVE else filename

groups = {}
for item in files:
    groups.setdefault(comparison_key(item["path"].name), []).append(item)

duplicate_groups = {name: items for name, items in groups.items() if len(items) > 1}
summary_rows = [
    {"nama_file": items[0]["path"].name, "jumlah": len(items)}
    for items in duplicate_groups.values()
]
detail_rows = [
    {"nama_file": item["path"].name, "path_lengkap": str(item["path"]),
     "ukuran_byte": item["size_bytes"], "ukuran_MB": round(item["size_bytes"] / (1024 ** 2), 2)}
    for items in duplicate_groups.values()
    for item in items
]

summary_df = pd.DataFrame(summary_rows, columns=["nama_file", "jumlah"])
detail_df = pd.DataFrame(detail_rows, columns=["nama_file", "path_lengkap", "ukuran_byte", "ukuran_MB"])
detail_df = detail_df.sort_values("ukuran_byte", ascending=False, ignore_index=True)

print(f"Folder: {TARGET_DIR.resolve()}")
print(f"Jumlah file yang berhasil diperiksa: {len(files)}")
print(f"Jumlah item yang tidak dapat diakses: {len(scan_errors)}")
print(f"Jumlah nama duplikat: {len(summary_df)}")
display(summary_df)
display(detail_df)

## 3. Simpan hasil (opsional)

Jika `EXPORT_CSV = True`, tabel detail file duplikat yang sudah diurutkan berdasarkan ukuran terbesar disimpan ke lokasi `OUTPUT_CSV`. Setel opsi sebelum menjalankan sel ini.

In [ ]:
if EXPORT_CSV:
    detail_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Hasil disimpan: {OUTPUT_CSV.resolve()}")
else:
    print("Ekspor CSV tidak aktif. Ubah EXPORT_CSV menjadi True jika diperlukan.")